#ETAPES A SUIVRENT
1. Supprimer les doublons
2. Supprimer les colonnes inutiles
3. Corriger les dtypes
4. Nettoyer les encodages incohérents
5. Traiter les valeurs aberrantes
6. Imputer les valeurs manquantes
7. Encoder les catégorielles
8. Normaliser/Standardiser
9. Train/Test split
10. Sauvegarder dans data/processed/

In [3]:
# Objectif : Nettoyer et préparer le dataset pour le modèle
# Input    : data/raw/churn_dataset_messy.csv
# Output   : data/processed/train.csv
#            data/processed/test.csv


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv("../data/raw/churn_dataset_messy.csv")
df_clean = df.copy()  # On ne touche jamais df original tres imporatnt pour le debug et a suite 

print(f"Shape original : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

Shape original : (15750, 18)
Colonnes : ['ID', 'CustomerID', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'RegistrationDate', 'SatisfactionScore', 'NumComplaints', 'Email', 'Exited']


In [5]:
# ÉTAPE 1 — SUPPRESSION DES DOUBLONS
# Détecté en EDA : 750 doublons (4.76%)
# Les doublons faussent les distributions et le modèle je mets les deux versions pour montrer l'importance de garder le df original pour le debug et la suite du projet

print("AVANT suppression doublons :", df_clean.shape)
print("Nombre de doublons :", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

print("APRÈS suppression doublons :", df_clean.shape)
print(f"Lignes supprimées : {df.shape[0] - df_clean.shape[0]}")

AVANT suppression doublons : (15000, 18)
Nombre de doublons : 0
APRÈS suppression doublons : (15000, 18)
Lignes supprimées : 750


In [6]:
# Colonnes identifiées en EDA comme non pertinentes pour le modèle
# ID, CustomerID : identifiants techniques
# Surname, Email : données personnelles
# RegistrationDate : 5045 valeurs uniques, inutilisable brut

cols_a_supprimer = ["ID", "CustomerID", "Surname", "Email", "RegistrationDate"]

df_clean = df_clean.drop(columns=cols_a_supprimer)

print(f"Colonnes supprimées : {cols_a_supprimer}")
print(f"Shape après suppression : {df_clean.shape}")
print(f"Colonnes restantes : {list(df_clean.columns)}")

Colonnes supprimées : ['ID', 'CustomerID', 'Surname', 'Email', 'RegistrationDate']
Shape après suppression : (15000, 13)
Colonnes restantes : ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'SatisfactionScore', 'NumComplaints', 'Exited']


In [7]:
# Colonnes détectées en EDA avec dtype incorrect
# Age, Balance, EstimatedSalary sont en object au lieu de numérique
# pd.to_numeric(errors='coerce') convertit les valeurs non numériques en NaN

cols_numeriques = ["CreditScore", "Age", "Tenure", "Balance", 
                   "NumOfProducts", "EstimatedSalary", 
                   "SatisfactionScore", "NumComplaints"]

for col in cols_numeriques:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Dtypes après conversion :")
print(df_clean[cols_numeriques].dtypes)
print(f"\nNaN générés par la conversion :")
print(df_clean[cols_numeriques].isna().sum())

Dtypes après conversion :
CreditScore          float64
Age                  float64
Tenure               float64
Balance              float64
NumOfProducts        float64
EstimatedSalary      float64
SatisfactionScore    float64
NumComplaints        float64
dtype: object

NaN générés par la conversion :
CreditScore          317
Age                  608
Tenure               443
Balance              485
NumOfProducts        305
EstimatedSalary      539
SatisfactionScore    871
NumComplaints        462
dtype: int64


In [8]:
# Geography — Germany/GERMANY/Allemagne, France/france/Frnace, Spain/Espagne
mapping_geo = {
    "Germany":"Germany", "GERMANY":"Germany", "Allemagne":"Germany",
    "France":"France",   "france":"France",   "Frnace":"France",
    "Spain":"Spain",     "Espagne":"Spain"
}
df_clean["Geography"] = df_clean["Geography"].map(mapping_geo)

# Gender — Male/MALE/M/Homme/1, Female/Femme/F/0/Unknown
mapping_gender = {
    "Male":"Male",   "MALE":"Male",   "M":"Male",   "Homme":"Male",   "1":"Male",
    "Female":"Female", "Femme":"Female", "F":"Female", "0":"Female",
    "Unknown": np.nan
}
df_clean["Gender"] = df_clean["Gender"].map(mapping_gender)

# HasCrCard — 0/1/Yes/No/True/False/Oui/Non/2
mapping_card = {
    1:"1", 0:"0", "1":"1", "0":"0",
    "Yes":"1", "No":"0", "True":"1", "False":"0",
    "Oui":"1", "Non":"0", 2:np.nan
}
df_clean["HasCrCard"] = df_clean["HasCrCard"].map(mapping_card).astype(float)

# IsActiveMember — 0/1/active/inactive/Y/N
mapping_active = {
    1:"1", 0:"0", "1":"1", "0":"0",
    "active":"1", "inactive":"0", "Y":"1", "N":"0"
}
df_clean["IsActiveMember"] = df_clean["IsActiveMember"].map(mapping_active).astype(float)

# SatisfactionScore — garder uniquement [1-5]
df_clean["SatisfactionScore"] = df_clean["SatisfactionScore"].where(
                                 df_clean["SatisfactionScore"].between(1, 5))

# NumOfProducts — garder uniquement [1-4]
df_clean["NumOfProducts"] = df_clean["NumOfProducts"].where(
                             df_clean["NumOfProducts"].between(1, 4))

# NumComplaints — remplacer 999 et -1 par NaN
df_clean["NumComplaints"] = df_clean["NumComplaints"].where(
                             df_clean["NumComplaints"].between(0, 10))


In [10]:

print("Vérification après nettoyage :")
print(f"Geography : {df_clean['Geography'].unique()}")
print(f"Gender : {df_clean['Gender'].unique()}")
print(f"HasCrCard : {df_clean['HasCrCard'].unique()}")
print(f"IsActiveMember : {df_clean['IsActiveMember'].unique()}")
print(f"SatisfactionScore : {sorted(df_clean['SatisfactionScore'].dropna().unique())}")
print(f"NumOfProducts : {sorted(df_clean['NumOfProducts'].dropna().unique())}")
print(f"NumComplaints : {sorted(df_clean['NumComplaints'].dropna().unique())}")

Vérification après nettoyage :
Geography : ['Germany' 'Spain' 'France' nan]
Gender : ['Male' 'Female' nan]
HasCrCard : [ 1.  0. nan]
IsActiveMember : [ 1.  0. nan]
SatisfactionScore : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
NumOfProducts : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
NumComplaints : [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


In [ ]:
# CreditScore — plage normale bancaire [300, 850]
# Valeurs détectées en EDA : min=-50, max=1500
df_clean["CreditScore"] = df_clean["CreditScore"].clip(lower=300, upper=850)

# Balance — Winsorizing borne sup détectée en EDA : 209 055
# Valeurs jusqu'à 1 milliard détectées - on plafonne
df_clean["Balance"] = df_clean["Balance"].clip(upper=209_055)

# EstimatedSalary — Winsorizing borne sup détectée en EDA : 236 792
df_clean["EstimatedSalary"] = df_clean["EstimatedSalary"].clip(upper=236_792)

# Tenure — valeur aberrante 100 détectée en EDA
# On garde uniquement [0, 10]
df_clean["Tenure"] = df_clean["Tenure"].where(
                     df_clean["Tenure"].between(0, 10))

# Age — valeurs aberrantes détectées en EDA [5.5, 73.5]
# On garde une plage réaliste pour un client bancaire
df_clean["Age"] = df_clean["Age"].where(
                  df_clean["Age"].between(18, 95))


In [12]:

print("Vérification après traitement aberrantes :")
print(f"CreditScore     : min={df_clean['CreditScore'].min():.0f}  max={df_clean['CreditScore'].max():.0f}")
print(f"Balance         : min={df_clean['Balance'].min():.0f}  max={df_clean['Balance'].max():.0f}")
print(f"EstimatedSalary : min={df_clean['EstimatedSalary'].min():.0f}  max={df_clean['EstimatedSalary'].max():.0f}")
print(f"Tenure          : min={df_clean['Tenure'].min():.0f}  max={df_clean['Tenure'].max():.0f}")
print(f"Age             : min={df_clean['Age'].min():.0f}  max={df_clean['Age'].max():.0f}")

print(f"\nNaN après traitement aberrantes :")
print(df_clean.isna().sum())

Vérification après traitement aberrantes :
CreditScore     : min=300  max=850
Balance         : min=-100000  max=209055
EstimatedSalary : min=-500  max=236792
Tenure          : min=0  max=10
Age             : min=18  max=94

NaN après traitement aberrantes :
CreditScore           317
Geography             223
Gender                630
Age                  1044
Tenure                738
Balance               485
NumOfProducts         537
HasCrCard             422
IsActiveMember        460
EstimatedSalary       539
SatisfactionScore    1744
NumComplaints         789
Exited                  0
dtype: int64


In [13]:
# Balance et EstimatedSalary — les valeurs négatives sont aberrantes
# Un solde bancaire et un salaire ne peuvent pas être négatifs
# On les remplace par NaN pour imputation ensuite

df_clean["Balance"] = df_clean["Balance"].where(df_clean["Balance"] >= 0)
df_clean["EstimatedSalary"] = df_clean["EstimatedSalary"].where(df_clean["EstimatedSalary"] >= 0)

print("Vérification après correction valeurs négatives :")
print(f"Balance         : min={df_clean['Balance'].min():.0f}  max={df_clean['Balance'].max():.0f}")
print(f"EstimatedSalary : min={df_clean['EstimatedSalary'].min():.0f}  max={df_clean['EstimatedSalary'].max():.0f}")
print(f"\nNaN Balance         : {df_clean['Balance'].isna().sum()}")
print(f"NaN EstimatedSalary : {df_clean['EstimatedSalary'].isna().sum()}")

Vérification après correction valeurs négatives :
Balance         : min=28  max=209055
EstimatedSalary : min=0  max=236792

NaN Balance         : 562
NaN EstimatedSalary : 616


## Étape 6 — Imputation des valeurs manquantes

In [16]:
# Stratégie d'imputation :
# Variables continues  - médiane (robuste aux outliers résiduels)
# Variables discrètes  - médiane
# Variables binaires   - mode (valeur la plus fréquente)
# Variables catégorielles - mode

# Continues et discrètes - médiane
cols_mediane = ["CreditScore", "Age", "Tenure", "Balance",
                "EstimatedSalary", "SatisfactionScore", "NumComplaints"]

for col in cols_mediane:
    mediane = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(mediane)
    print(f"{col:20} - imputé avec médiane = {mediane:.1f}")

# NumOfProducts - mode (valeur discrète avec distribution asymétrique)
mode_products = df_clean["NumOfProducts"].mode()[0]
df_clean["NumOfProducts"] = df_clean["NumOfProducts"].fillna(mode_products)
print(f"{'NumOfProducts':20} - imputé avec mode = {mode_products:.0f}")

# Binaires - mode
for col in ["HasCrCard", "IsActiveMember"]:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"{col:20} - imputé avec mode = {mode_val:.0f}")

# Catégorielles - mode
for col in ["Geography", "Gender"]:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"{col:20} - imputé avec mode = {mode_val}")

print(f"\nNaN restants après imputation :")
print(df_clean.isna().sum())
print(f"\nShape : {df_clean.shape}")

CreditScore          - imputé avec médiane = 651.0
Age                  - imputé avec médiane = 40.0
Tenure               - imputé avec médiane = 5.0
Balance              - imputé avec médiane = 63496.0
EstimatedSalary      - imputé avec médiane = 99817.7
SatisfactionScore    - imputé avec médiane = 3.0
NumComplaints        - imputé avec médiane = 0.0
NumOfProducts        - imputé avec mode = 1
HasCrCard            - imputé avec mode = 1
IsActiveMember       - imputé avec mode = 1
Geography            - imputé avec mode = Germany
Gender               - imputé avec mode = Male

NaN restants après imputation :
CreditScore          0
Geography            0
Gender               0
Age                  0
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
SatisfactionScore    0
NumComplaints        0
Exited               0
dtype: int64

Shape : (15000, 13)


### Étape 7 — Encodage des variables catégorielles

In [17]:
# Geography → One-Hot Encoding
# On utilise pd.get_dummies car pas de relation ordinale entre les pays
# drop_first=True pour éviter la multicolinéarité
# France devient la référence (supprimée)

df_clean = pd.get_dummies(df_clean, columns=["Geography"], drop_first=True)

# Gender → Label Encoding
# Male=1, Female=0
df_clean["Gender"] = (df_clean["Gender"] == "Male").astype(int)

# Convertir les colonnes booléennes générées par get_dummies en int
bool_cols = df_clean.select_dtypes(include="bool").columns
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

print("Colonnes après encodage :")
print(list(df_clean.columns))
print(f"\nShape : {df_clean.shape}")
print(f"\nDtypes :")
print(df_clean.dtypes)

Colonnes après encodage :
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'SatisfactionScore', 'NumComplaints', 'Exited', 'Geography_Germany', 'Geography_Spain']

Shape : (15000, 14)

Dtypes :
CreditScore          float64
Gender                 int64
Age                  float64
Tenure               float64
Balance              float64
NumOfProducts        float64
HasCrCard            float64
IsActiveMember       float64
EstimatedSalary      float64
SatisfactionScore    float64
NumComplaints        float64
Exited                 int64
Geography_Germany      int64
Geography_Spain        int64
dtype: object


#### Étape 8 — Train/Test Split + Standardisation

In [ ]:
# Séparation features / cible
X = df_clean.drop(columns=["Exited"])
y = df_clean["Exited"]

# Train/Test split — 80/20
# stratify=y pour conserver le ratio de churn dans les deux sets
# random_state=42 pour la reproductibilité

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Taux churn train : {y_train.mean()*100:.1f}%")
print(f"Taux churn test  : {y_test.mean()*100:.1f}%")


X_train : (12000, 13)
X_test  : (3000, 13)
Taux churn train : 26.2%
Taux churn test  : 26.2%


In [19]:

# Standardisation — uniquement sur les variables continues
# On standardise APRÈS le split pour éviter le data leakage
# Le scaler est fitté sur train et appliqué sur train ET test
cols_a_scaler = ["CreditScore", "Age", "Tenure", 
                 "Balance", "EstimatedSalary"]

scaler = StandardScaler()
X_train[cols_a_scaler] = scaler.fit_transform(X_train[cols_a_scaler])
X_test[cols_a_scaler]  = scaler.transform(X_test[cols_a_scaler])

print(f"\nAprès standardisation :")
print(X_train[cols_a_scaler].describe().round(2))


Après standardisation :
       CreditScore       Age    Tenure   Balance  EstimatedSalary
count     12000.00  12000.00  12000.00  12000.00         12000.00
mean         -0.00     -0.00      0.00      0.00            -0.00
std           1.00      1.00      1.00      1.00             1.00
min          -3.47     -1.98     -1.63     -1.49            -2.08
25%          -0.64     -0.64     -0.99     -0.80            -0.69
50%           0.01     -0.01     -0.01     -0.15            -0.02
75%           0.68      0.61      0.96      0.63             0.66
max           1.99      4.81      1.61      2.90             2.81


#### Étape 9 — Sauvegarde

In [20]:
import os

os.makedirs("../data/processed", exist_ok=True)

# Reconstruction des datasets complets avec la cible
train = X_train.copy()
train["Exited"] = y_train.values

test = X_test.copy()
test["Exited"] = y_test.values

train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv",  index=False)

print(f"train.csv sauvegardé : {train.shape}")
print(f"test.csv  sauvegardé : {test.shape}")
print(f"\nColonnes finales : {list(train.columns)}")

train.csv sauvegardé : (12000, 14)
test.csv  sauvegardé : (3000, 14)

Colonnes finales : ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'SatisfactionScore', 'NumComplaints', 'Geography_Germany', 'Geography_Spain', 'Exited']


# Conclusions Générales — Preprocessing ChurnGuard

## Résumé des transformations

**Doublons** : 750 supprimés → 15 750 à 15 000 lignes

**Colonnes supprimées** : ID, CustomerID, Surname, Email, RegistrationDate → 18 à 13 colonnes

**Dtypes** : Age, Balance, EstimatedSalary et autres convertis en numérique via pd.to_numeric(errors='coerce')

**Encodages incohérents** : Geography, Gender, HasCrCard, IsActiveMember, SatisfactionScore, NumOfProducts, NumComplaints nettoyés et mappés vers des valeurs cohérentes

**Valeurs aberrantes** : CreditScore clippé [300, 850], Balance et EstimatedSalary Winsorisés, Tenure et Age filtrés via where(), valeurs négatives supprimées

**Imputation** : médiane pour toutes les continues et discrètes, mode pour les binaires et catégorielles → 0 NaN restants

**Encodage catégoriel** : Geography encodé en One-Hot (Geography_Germany, Geography_Spain), Gender en Label Encoding (Male=1, Female=0)

**Split** : 80/20 stratifié random_state=42 → 12 000 train / 3 000 test, taux churn conservé à 26.2% dans les deux sets

**Standardisation** : StandardScaler fitté sur train uniquement pour éviter le data leakage, appliqué sur train et test → mean=0, std=1

## Prochaine étape

05_model_training.ipynb : entraînement des modèles Logistic Regression, Decision Tree, Random Forest avec MLflow tracking.